# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [1]:
from google import genai
from kaggle_secrets import UserSecretsClient

# 1. Configuración de la clave
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GEMINI_API_KEY")

# 2. Inicializar cliente
client = genai.Client(api_key=api_key)

In [2]:
# 3. Llamada con el nombre de modelo completo (models/...)
try:
    response = client.models.generate_content(
        model="models/gemini-2.5-flash-lite", 
        contents="Explica brevemente cómo funciona la IA."
    )
    print(response.text)
except Exception as e:
    print(f"Error: {e}")

La IA funciona esencialmente **permitiendo a las computadoras "aprender" y "pensar" de manera similar a los humanos, pero a una escala y velocidad mucho mayores.**

Aquí tienes una explicación breve:

1.  **Datos (Entrada):** La IA necesita grandes cantidades de información (datos) para aprender. Estos datos pueden ser imágenes, texto, sonidos, números, etc.
2.  **Algoritmos (Aprendizaje):** Se utilizan algoritmos, que son conjuntos de instrucciones, para procesar estos datos. Estos algoritmos buscan patrones, relaciones y reglas dentro de la información.
3.  **Modelos (Conocimiento):** A través del aprendizaje, la IA crea "modelos". Un modelo es como un mapa mental que representa el conocimiento adquirido. Por ejemplo, un modelo de reconocimiento de imágenes habrá aprendido a identificar características de diferentes objetos.
4.  **Predicción/Acción (Salida):** Una vez entrenada, la IA puede usar su modelo para hacer predicciones, tomar decisiones o realizar acciones basadas en nuevos

## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [3]:
from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
newsgroupsdocs = newsgroups.data

In [4]:
import pandas as pd
df = pd.DataFrame(newsgroupsdocs, columns=['text'])
df

,text
0,\n\nI am sure some bashers of Pens fans are pr...
1,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...
4,1) I have an old Jasmine drive which I cann...
...,...
18841,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...
18842,\nNot in isolated ground recepticles (usually ...
18843,I just installed a DX2-66 CPU in a clone mothe...
18844,\nWouldn't this require a hyper-sphere. In 3-...


### 2.2 Transformo a embeddings

In [5]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,text,text_norm
0,\n\nI am sure some bashers of Pens fans are pr...,I am sure some bashers of Pens fans are pretty...
1,My brother is in the market for a high-perform...,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...,Finally you said what you dream about. Mediter...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,Think! It's the SCSI card doing the DMA transf...
4,1) I have an old Jasmine drive which I cann...,1) I have an old Jasmine drive which I cannot ...


In [6]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  I am sure some bashers of Pens fans are pretty...
 1       1         0  My brother is in the market for a high-perform...
 2       2         0  Finally you said what you dream about. Mediter...
 3       2         1  urds and Turks once upon a time! Ohhhh so swed...
 4       3         0  Think! It's the SCSI card doing the DMA transf...,
 38871)

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

2026-01-07 17:22:01.288100: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767806521.531901      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767806521.600926      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767806522.191484      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767806522.191519      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767806522.191522      55 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [8]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/2430 [00:00<?, ?it/s]

In [9]:
print(embeddings.shape, embeddings.dtype)

(38871, 768) float32


In [19]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Building homemade batteries"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

### 2.3 Creo una query y hago la búsqueda

In [20]:
# Calculamos la similitud (producto punto) entre la query y todos los embeddings
# Como los vectores ya están normalizados, esto equivale a la similitud coseno
scores = np.dot(embeddings, query_vec.T).flatten()

# Obtenemos los índices de los 5 resultados con mayor puntuación
top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]

print(f"Búsqueda finalizada para: '{query_text}'")

Búsqueda finalizada para: 'Building homemade batteries'


Obtengo los 5 documentos más similares a mi query

In [21]:
print(f"Resultados más relevantes para: {query_text}\n")

for i, idx in enumerate(top_indices):
    score = scores[idx]
    chunk_text = chunks_df.iloc[idx]["text"]
    doc_id = chunks_df.iloc[idx]["doc_id"]
    
    print(f"{i+1}. [Similitud: {score:.4f}] (Doc ID: {doc_id})")
    print(f"Texto: {chunk_text}")
    print("-" * 80)

Resultados más relevantes para: Building homemade batteries

1. [Similitud: 0.8709] (Doc ID: 4797)
Texto: : My 9 yr old son has signed up to do a science report on batteries. I was : wondering if anyone could provide me with some information as to how to : construct a home-built battery. In my grade school days, I remember seeing : the 'ice cube tray' version, but I don't remember what to use as a good : electrolyte or what the easily obtainable metals were. : : Thank you in advance. I remember watching a whole "Mr. Wizzard" program on this subject when I was a kid. The battery constructed on the program which made the biggest impression on me, and generated the most power, was made using a galvanized bucket (for the zinc) and a copper toilet tank float. The electrolyte was sauerkraut!
--------------------------------------------------------------------------------
2. [Similitud: 0.8694] (Doc ID: 16678)
Texto: : My 9 yr old son has signed up to do a science report on batteries. I was :

In [22]:
# Construimos el contexto con los fragmentos recuperados
context = "\n\n".join(chunks_df.iloc[top_indices]["text"].tolist())
context

prompt = f"""
Eres un asistente experto. Utiliza la siguiente información extraída de un corpus de noticias 
para dar un resumen de los resultados. Si la información no es suficiente, indícalo.

CONTEXTO:
{context}

PREGUNTA DEL USUARIO:
{query_text}

RESPUESTA:
"""

# Usamos el cliente de Gemini configurado en la parte 1
response = client.models.generate_content(
    model="models/gemini-2.5-flash-lite",
    contents=prompt
)

print("--- RESPUESTA GENERADA POR GEMINI ---")
print(response.text)

--- RESPUESTA GENERADA POR GEMINI ---
Aquí tienes un resumen de los resultados sobre la construcción de baterías caseras, basado en la información proporcionada:

**Componentes básicos:**

*   **Metales:**
    *   **Cobre:** Se menciona el uso de cobre en forma de alambre o flotadores de cobre.
    *   **Zinc:** Se sugiere usar tiras de zinc (disponibles en ferreterías para evitar el musgo en tejados) o materiales galvanizados (recubiertos de zinc) como clavos o cubos. Se advierte que las monedas actuales podrían ser principalmente de zinc.
*   **Electrolito (sustancia que permite el paso de la corriente):**
    *   **Líquidos ácidos:** Se mencionan el jugo de limón (ácido cítrico) y la solución ácida en general.
    *   **Alimentos fermentados:** Se cita la *chucrut* (sauerkraut) como electrolito en una versión de batería casera.
    *   **Agua con ácido:** Implícito en el uso de discos de papel empapados en ácido.

**Métodos y ejemplos:**

*   **Batería de "bandeja de cubitos de hiel